# R3 — LangChain structured extraction: waiting-period facts

Reads the two waiting-period policy documents (`02_Waiting_Periods_Individual_Plans.md`, `03_Waiting_Periods_Group_Plans.md`) straight from the landing volume, drives the same extraction through **two** LangChain chat wrappers — `ChatDatabricks` (the Llama endpoint) and `ChatOpenAI` (pointed at the same Databricks serving endpoint via its OpenAI-compatible route) — and parses the output with a Pydantic model `WaitingPeriodFacts` + `PydanticOutputParser`.

Deliverable: `GenAi-1/evidence/r3_langchain.json` with keys `chat_databricks_reply`, `chat_openai_reply` (both non-empty text) and `waiting_facts` (two objects, document 02 first, then 03).

Expected facts (verified from the docs themselves):
- 02 individual: initial waiting 30 days, PED waiting 36 months
- 03 group: initial waiting 0 days, PED waiting 12 months

In [0]:
%pip install langchain-core langchain-databricks langchain-openai "protobuf>=6,<7" "pyarrow>=21" "packaging>=25" "idna>=3.11" "mlflow-skinny>=3.11,<4" --quiet

In [0]:
# notebook-scoped install: restart Python so the fresh packages become visible (serverless does not auto-restart)
dbutils.library.restartPython()

In [0]:
# Paper 4 config cell - change endpoint names HERE and nowhere else
CATALOG = "claims"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"

print(f"CATALOG={CATALOG} | CHAT_MODEL={CHAT_MODEL} | EMBED_MODEL={EMBED_MODEL}")

In [0]:
# 1. Load the two waiting-period policy documents (full markdown, straight from the landing volume)
from pyspark.sql import functions as F

DOCS = ["02_Waiting_Periods_Individual_Plans.md", "03_Waiting_Periods_Group_Plans.md"]

docs_df = (
    spark.read.format("binaryFile")
    .load("/Volumes/claims/raw/landing/genai/policy_docs/")
    .withColumn("doc_name", F.regexp_extract("path", r"[^/]+$", 0))
    .withColumn("raw", F.trim(F.col("content").cast("string")))
    .filter(F.col("doc_name").isin(DOCS))
)
DOC_TEXT = {r.doc_name: r.raw for r in docs_df.collect()}

assert set(DOC_TEXT.keys()) == set(DOCS), f"Expected exactly the two waiting-period docs, got {sorted(DOC_TEXT)}"
for name in DOCS:
    assert "Applies to:" in DOC_TEXT[name], f"{name} is missing its 'Applies to:' line"
    print(f"{name}: {len(DOC_TEXT[name]):,} chars, 'Applies to' line present")

In [0]:
# 2. Pydantic model + output parser (R3 contract)
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class WaitingPeriodFacts(BaseModel):
    """Waiting-period facts for one policy document (one plan family)."""

    doc_name: str = Field(description="The policy document file name exactly, with no folder, e.g. '02_Waiting_Periods_Individual_Plans.md'")
    plan_family: str = Field(description="The plan family the document's 'Applies to:' line declares: 'individual' or 'group'")
    initial_waiting_days: int = Field(description="Initial waiting period in days for that plan family (0 if the document says there is none)")
    ped_waiting_months: int = Field(description="Pre-existing disease (PED) waiting period in months for that plan family")


parser = PydanticOutputParser(pydantic_object=WaitingPeriodFacts)
print(parser.get_format_instructions())

In [0]:
# 3. One prompt, run through both wrappers (R3: ChatDatabricks AND ChatOpenAI)
from langchain_core.prompts import PromptTemplate

PROMPT = PromptTemplate(
    template="""Extract the waiting-period facts from this Saffron Shield policy document.

{format_instructions}

Rules:
- doc_name must be exactly the file name below, with no folder path.
- plan_family must be the family the document's 'Applies to:' line declares (individual or group), not one guessed from the file name.
- Read the numbers from the document itself. If the document says there is NO initial waiting period, use 0.

File name: {doc_name}

Document:
{document}""",
    input_variables=["doc_name", "document"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

def run_chain(chat_model):
    """Run the extraction prompt through one chat model; return (reply_text, parsed_facts)."""
    reply_text = ""
    facts = []
    for doc in DOCS:
        msg = PROMPT.invoke({"doc_name": doc, "document": DOC_TEXT[doc]})
        out = chat_model.invoke(msg)
        reply_text += out.content if isinstance(out.content, str) else str(out.content)
        facts.append(parser.parse(out.content))
    return reply_text, facts

# ground-truth from the docs themselves, used for self-checks in step 5
EXPECTED = {
    "02_Waiting_Periods_Individual_Plans.md": {"plan_family": "individual", "initial_waiting_days": 30, "ped_waiting_months": 36},
    "03_Waiting_Periods_Group_Plans.md": {"plan_family": "group", "initial_waiting_days": 0, "ped_waiting_months": 12},
}
print("Prompt ready; ground truth loaded for both docs.")

In [0]:
# 4a. ChatDatabricks wrapper - the Llama endpoint from the config cell
from langchain_databricks import ChatDatabricks

chat_databricks = ChatDatabricks(endpoint=CHAT_MODEL, max_tokens=1024, temperature=0)
chat_databricks_reply, facts_databricks = run_chain(chat_databricks)

print(f"reply length: {len(chat_databricks_reply):,} chars (non-empty: {bool(chat_databricks_reply.strip())})")
for f in facts_databricks:
    print(f)

In [0]:
# 4b. ChatOpenAI wrapper - pointed at the SAME Databricks serving endpoint via its OpenAI-compatible route
#     No OpenAI account involved: Databricks serving endpoints accept OpenAI-API-format requests.
#     Notes: (1) the workspace token comes from the SDK's credentials provider (federated auth - config.token is None)
#            (2) do NOT set max_tokens here: current langchain-openai maps it to 'max_completion_tokens',
#                which the Databricks route rejects with BAD_REQUEST 'unknown field'
from databricks.sdk import WorkspaceClient
from langchain_openai import ChatOpenAI

_cfg = WorkspaceClient().config
_bearer = _cfg.authenticate()["Authorization"].split(" ", 1)[1]
chat_openai = ChatOpenAI(
    model=CHAT_MODEL,
    base_url=f"{_cfg.host}/serving-endpoints",
    api_key=_bearer,
    temperature=0,
)
chat_openai_reply, facts_openai = run_chain(chat_openai)

print(f"reply length: {len(chat_openai_reply):,} chars (non-empty: {bool(chat_openai_reply.strip())})")
for f in facts_openai:
    print(f)

In [0]:
# 5. Assemble evidence dict, self-check against ground truth, print the JSON preview
import json

# waiting_facts: list of TWO objects, document 02 first, then 03 (use the ChatDatabricks parse)
waiting_facts = [WaitingPeriodFacts(**f.model_dump()) for f in facts_databricks]
waiting_facts.sort(key=lambda f: f.doc_name)  # 02_... sorts before 03_...

r3_evidence = {
    "chat_databricks_reply": chat_databricks_reply,
    "chat_openai_reply": chat_openai_reply,
    "waiting_facts": [f.model_dump() for f in waiting_facts],
}

# --- R3 contract self-checks ---
assert bool(r3_evidence["chat_databricks_reply"].strip()), "chat_databricks_reply must be non-empty"
assert bool(r3_evidence["chat_openai_reply"].strip()), "chat_openai_reply must be non-empty"
assert [f["doc_name"] for f in r3_evidence["waiting_facts"]] == DOCS, "waiting_facts must be [02, 03] in that order"
for f in r3_evidence["waiting_facts"]:
    assert f["doc_name"] == f["doc_name"].split("/")[-1], "doc_name must carry no folder"
    assert f["plan_family"] in ("individual", "group"), f"bad plan_family: {f['plan_family']}"
    exp = EXPECTED[f["doc_name"]]
    for k in ("plan_family", "initial_waiting_days", "ped_waiting_months"):
        assert f[k] == exp[k], f"{f['doc_name']}.{k}: got {f[k]!r}, expected {exp[k]!r}"

# both wrapper runs must agree with the docs (cross-check between the two wrappers)
assert [f.model_dump() for f in facts_databricks] == [f.model_dump() for f in facts_openai], "wrappers disagree on parsed facts"

print(json.dumps(r3_evidence, indent=2)[:1200])
print("...")
print("ALL R3 SELF-CHECKS PASSED")